In [1]:
# IMPORTS
import csv
from pathlib import Path

In [2]:
# SETTINGS 
term = "f2026"
course = "3321"
week = 1

data_folder = Path("data/participation")
corrections_path = data_folder / "name_corrections.csv"


classlist_files = {
    "2320": f"mos2320_classlist_{term}.csv",
    "3321": f"mos3321_classlist_{term}.csv"
}

gradeexport_files = {
    "2320": f"mos2320_gradeexport_{term}.csv",
    "3321": f"mos3321_gradeexport_{term}.csv"
}


In [3]:
##### FUNCTIONS 

def read_attendance(filename): 
    file_path = data_folder / filename

    with open(file_path, "r", encoding="utf-8-sig", newline="") as file: 
        reader = csv.reader(file)
        rows = list(reader)

    student_names = []

    for row in rows[1:]:
        if row: 
            name = row[0].strip().lower()

            if name: 
                student_names.append(name)

    return set(student_names)

def normalize_name(name):
    if "," in name: 
        last_name, first_name = name.split(",", 1)
        name = first_name + " " + last_name

    return " ".join(name.lower().split())

def simple_name(name):
    normalized = normalize_name(name)
    name_parts = normalized.split()

    first_name = name_parts[0]
    last_name = name_parts[-1]

    return first_name + " " + last_name

def match_student(name):
    simplified = simple_name(name)

    if simplified in name_corrections:
        student_id = name_corrections[simplified]
        return students_by_id.get(student_id)

    matches = student_lookup.get(simplified, [])

    if len(matches) == 1:
        return matches[0]

    return None

def load_name_corrections(course):
    corrections = {}

    with open(corrections_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            if row["Course"] == course:
                corrections[row["PollEverywhereName"]] = row["OrgDefinedId"]

    return corrections


def get_meeting_number(filename):
    week_and_meeting = Path(filename).stem.split("-")[-1]

    if "." in week_and_meeting:
        return int(week_and_meeting.split(".")[-1])

    return 1


def get_attending_students(filename):
    attendance = read_attendance(filename)

    attending_ids = set()
    unmatched_names = []

    for name in attendance:
        student = match_student(name)

        if student is not None:
            attending_ids.add(student["OrgDefinedId"])
        else:
            unmatched_names.append(name)

    return attending_ids, unmatched_names


In [4]:
# READ FILES 



name_corrections = load_name_corrections(course)

print(f"Course {course}: {len(name_corrections)} saved corrections")


classlist_path = data_folder / classlist_files[course]

with open(classlist_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    classlist = list(reader)

students = [
    row for row in classlist
    if row["Role"] == "Learner"
]

print(f"Course {course}: {len(students)} students")


student_lookup = {}
students_by_id = {}

for student in students:
    name = simple_name(student["Name"])
    student_id = student["OrgDefinedId"]

    if name not in student_lookup:
        student_lookup[name] = []

    student_lookup[name].append(student)
    students_by_id[student_id] = student

print("Students indexed by ID:", len(students_by_id))
print("Unique simplified names:", len(student_lookup))

ambiguous_names = {
    name: matches
    for name, matches in student_lookup.items()
    if len(matches) > 1
}

print("Ambiguous Names:", len(ambiguous_names))

for name, matches in ambiguous_names.items():
    print(name, ":", len(matches), "students")


grades_path = data_folder / gradeexport_files[course]

with open(grades_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    grade_headers = reader.fieldnames
    grade_rows = list(reader)

print(f"Course {course}: {len(grade_rows)} grade records")



attendance_files = sorted(
    file for file in data_folder.glob(f"{course}-*-w*.csv")
    if file.stem.split("-")[-1].split(".")[0] == f"w{week}"
)

if not attendance_files:
    raise FileNotFoundError(
        f"No attendance files found for course {course}, week {week}"
    )

for file in attendance_files:
    print(file.name)




Course 3321: 3 saved corrections
Course 3321: 78 students
Students indexed by ID: 78
Unique simplified names: 78
Ambiguous Names: 0
Course 3321: 78 grade records
3321-550-w1.csv
3321-551-w1.csv


In [5]:
# MATCH & REVIEW ATTENDANCE 


course_attendance = {}
all_unmatched = set()

for file in attendance_files:
    attending_ids, unmatched_names = get_attending_students(file.name)

    course_attendance[file.name] = attending_ids
    all_unmatched.update(unmatched_names)

    print(f"{file.name}: {len(attending_ids)} matched students")

print("\nUnmatched names:", sorted(all_unmatched))


if all_unmatched:
    raise ValueError(
        f"{len(all_unmatched)} unmatched names need to be resolved before grading."
    )

print("All attendance names matched. Ready to calculate grades.")





3321-550-w1.csv: 34 matched students
3321-551-w1.csv: 32 matched students

Unmatched names: []
All attendance names matched. Ready to calculate grades.


In [6]:
# CALCULATE PARTICIPATION GRADES 


weekly_attendance = {}

for filename, attending_ids in course_attendance.items():
    meeting = get_meeting_number(filename)

    if meeting not in weekly_attendance:
        weekly_attendance[meeting] = set()

    weekly_attendance[meeting].update(attending_ids)

for meeting, attending_ids in sorted(weekly_attendance.items()):
    print(f"Meeting {meeting}: {len(attending_ids)} unique students")


weekly_scores = {}

for student in students:
    student_id = student["OrgDefinedId"]

    score = 0

    for attending_ids in weekly_attendance.values():
        if student_id in attending_ids:
            score += 1

    weekly_scores[student_id] = score

print("Students graded:", len(weekly_scores))

for score in sorted(set(weekly_scores.values())):
    count = list(weekly_scores.values()).count(score)
    print(f"{score} points: {count} students")


grade_column = next(
    header for header in grade_headers
    if header.startswith(f"Week {week} Points Grade <")
)

print(grade_column)



Meeting 1: 66 unique students
Students graded: 78
0 points: 12 students
1 points: 66 students
Week 1 Points Grade <Numeric MaxPoints:1 Weight:11.11111111 Category:Participation CategoryWeight:10>


In [7]:
# VALIDATE & EXPORT 


classlist_ids = {
    student_id.lstrip("#")
    for student_id in weekly_scores
}

grade_export_ids = {
    row["OrgDefinedId"].lstrip("#")
    for row in grade_rows
}

print("Students in class list:", len(classlist_ids))
print("Students in grade export:", len(grade_export_ids))

print("Missing from grade export:", len(classlist_ids - grade_export_ids))
print("Missing from class list:", len(grade_export_ids - classlist_ids))


import_rows = []

for row in grade_rows:
    student_id = row["OrgDefinedId"]
    lookup_id = student_id.lstrip("#")

    score = weekly_scores[lookup_id]

    import_rows.append({
        "OrgDefinedId": student_id,
        grade_column: score,
        "End-of-Line Indicator": row["End-of-Line Indicator"]
    })

print("Students prepared for import:", len(import_rows))
print("Total points awarded:", sum(row[grade_column] for row in import_rows))


import_scores = [row[grade_column] for row in import_rows]

print("Total students:", len(import_rows))

for score in range(len(weekly_attendance) + 1):
    print(f"{score} points:", import_scores.count(score))

print("Total points awarded:", sum(import_scores))


output_folder = Path("outputs/participation")
output_folder.mkdir(exist_ok=True)

output_path = output_folder / f"{course}_week{week}_participation.csv"

import_headers = [
    "OrgDefinedId",
    grade_column,
    "End-of-Line Indicator"
]

with open(output_path, "w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=import_headers)

    writer.writeheader()
    writer.writerows(import_rows)

print("Created:", output_path.resolve())





Students in class list: 78
Students in grade export: 78
Missing from grade export: 0
Missing from class list: 0
Students prepared for import: 78
Total points awarded: 66
Total students: 78
0 points: 12
1 points: 66
Total points awarded: 66
Created: C:\Users\klbar\OneDrive - The University of Western Ontario\Faculty\Teaching Infrastructure\outputs\participation\3321_week1_participation.csv


In [8]:
# ATTENDANCE SUMMARY 

In [9]:

print("Course:", course)
print("Week:", week)
print("Students:", len(import_rows))
print("Maximum score:", max(import_scores))
print("Total points awarded:", sum(import_scores))
print("Unmatched names:", len(all_unmatched))


Course: 3321
Week: 1
Students: 78
Maximum score: 1
Total points awarded: 66
Unmatched names: 0
